### Installation

In [6]:
%%capture
!pip uninstall unsloth unsloth_zoo -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth-zoo.git"


In [1]:
from unsloth import FastModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    dtype = None, # None for auto detection
    max_seq_length = 1024, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

## Data preprocess

In [ ]:
!git clone --recurse-submodules https://github.com/hasbai/hcf
%cd hcf

In [8]:
# pair: 167k 单 QA 对
!python prepare_dataset.py --mode pair --output hcf_sft_pair.jsonl

# window: 52k 多轮窗口
!python prepare_dataset.py --mode window --output hcf_sft_window.jsonl

# 混合（推荐用这个训练）
import json, random
random.seed(42)
lines = open('hcf_sft_pair.jsonl').readlines() + open('hcf_sft_window.jsonl').readlines()
random.shuffle(lines)
open('hcf_sft_mixed.jsonl', 'w').writelines(lines)
print(f"总样本数: {len(lines)}")


Mode:              pair
Files processed:   489
Cleaned turns:     391158
Examples written:  167275
Total content:     17.5 M chars
Output:            hcf_sft_pair.jsonl
Mode:              window
Files processed:   489
Cleaned turns:     391158
Examples written:  52262
Total content:     20.1 M chars
Output:            hcf_sft_window.jsonl
总样本数: 219537


# Let's finetune Gemma 4!

You can finetune the vision and text parts for now through selection - the audio part can also be finetuned - we're working to make it selectable as well!

We now add LoRA adapters so we only need to update a small amount of parameters!

In [11]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [9]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from unsloth.chat_templates import standardize_data_formats

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)
dataset = load_dataset("json", data_files="hcf_sft_pair.jsonl", split="train")
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        .removeprefix("<bos>")
        for c in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:400])

Generating train split: 0 examples [00:00, ? examples/s]

Unsloth: Standardizing formats (num_proc=3):   0%|          | 0/167275 [00:00<?, ? examples/s]

Map:   0%|          | 0/167275 [00:00<?, ? examples/s]

<|turn>user
所以说啊，这个……<turn|>
<|turn>model
哎，我我哎，我不知道说什么了。我说我不知道说什么，不是我不知道说什么了，就是说我说我不知道说什么。你看，哎呀，你们，我不知道说什么了的意思是我不知道应该去说什么，而不是说我不知道说什么，你能明白这个意思吗？我就是说你们冷静一下，卖下抠字的，冷静一下，冷静一下。我开播，我开播怎么了？我是个小人物啊，是不是？我开个播挣点钱不寒碜，是不是？开个播挣点钱怎么了？我是吧，那是谁？是吧，我我该开该该该开开我的是吧。今天是可以，今天不说，不说，喝喝喝水，喝水，喝水，你们都渴了吧？喝口水，喝口水，别扣了，别扣了，喝口水，喝口水，憋的属实难受。憋的属实难受，我跟你们讲，你们难受我也难受。刚开不少了，我刚开能有五分钟，五分钟刚开五分钟，对，两块钱的水是。两块钱的水，你看我这两块钱的水是吧？都没话题聊了，今天真没话题聊了，连矿泉


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [16]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 2000,
        learning_rate = 2e-4,
        packing = True, # 样本平均长度远小于context window
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        dataloader_num_workers = 4,
        dataloader_pin_memory = True,
    ),
)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/167275 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/167275 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/167275 [00:00<?, ? examples/s]

In [19]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
11.484 GB of memory reserved.


# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 167,275 | Num Epochs = 1 | Total steps = 2,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 18,350,080 of 8,014,506,528 (0.23% trained)


Step,Training Loss
50,4.014048
100,3.930694
150,3.905855
200,3.835226
250,3.745875
300,3.756711
350,3.765006
400,3.748882
450,3.679104
500,3.690063


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-4` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "电车就买什么",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

The sky appears blue primarily due to a phenomenon called **Rayleigh scattering**. Here's a breakdown of how this works:

1. **Sunlight is composed of different wavelengths:** Sunlight, which comes from the sun, is made up of various colors, each with a different wavelength. Blue light has a shorter wavelength


### Save GGUF to HF
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
from google.colab import userdata

model.push_to_hub_gguf(
    "hasbai/hcf-gemma-4",
    tokenizer,
    quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
    token = userdata.get('hfKey'),
)